# soyabean_price_forecast - run & understand the pipeline

This notebook walks the project **the way the data flows**, so you can watch each stage happen:

```
   collectors/            store (parquet)          features/build.py        data/features/
raw APIs ----------> data/staged/*.parquet ----------------------------> training_weekly.csv
(weather, markets,     one table per source,     one row per WEEK,          timellm_...csv
 mandi, USDA) +        obs_date + available_at    point-in-time features     data_dictionary.csv
 agent (Tavily+LLM)
```

**Three stages:** (1) collect structured sources, (2) optionally enrich with the research agent,
(3) build the weekly training table. Run the cells top to bottom.

> Keep this notebook in the project root (the folder that has `src/`, `config/`).
> First time only, install deps:  `pip install -r requirements.txt`

## 0 - Setup: make the package importable and load settings

In [ ]:
import sys, os, logging
from pathlib import Path

# Find the project root (the folder that contains src/soyfc) and make `soyfc` importable.
here = Path.cwd()
ROOT = next((p for p in [here, *here.parents] if (p / "src" / "soyfc").exists()), here)
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")

from soyfc.config import load_settings
from soyfc import store            # I/O layer: store.read(s, ...) / store.upsert(s, ...)

s = load_settings()                # reads config/settings.yaml  (+ .env for API keys)
print("Project root :", ROOT)
print("Target       :", s["target"]["market"], s["target"]["commodity"])
print("History      :", s.start_date, "->", s.end_date)
# Each market's raw/staged/features live under their own folder, so switching markets is safe.
print("Work folder  :", s.work_dir, "(per-target: data/<market>_<commodity>)")

## 1 - What are we building? Read the config

`config/settings.yaml` drives everything. `Settings` (in `src/soyfc/config.py`) just wraps that
YAML, with a few shortcuts like `s.start_date` and `s.week_rule`.

In [ ]:
from soyfc.agent.runner import TASKS

t = s["target"]
print("Forecast horizons (weeks):", t["horizons_weeks"])
print("Weekly bucket            :", s.week_rule, "(weeks end on Friday; cutoff = Saturday)")
print("Mandi sources, in order  :", s["mandi_sources"]["order"])
print("Peer mandis              :", [p["market"] for p in s.get("peer_markets", [])])
print()
print("Agent research areas:")
for name, task in TASKS.items():
    print("  - %-16s %s" % (name, task.description[:70]))

## 2 - Stage 1: COLLECT (watch one collector run)

Every collector has the same shape: `collect(s)` -> fetch -> clean -> set `available_at` ->
`store.upsert(...)`. **Weather** is the easiest to watch because Open-Meteo needs no API key.

The two lines below *trim* it (2 points, short history) so this runs in well under a minute.
Delete them to collect the full catchment from `settings.yaml`.

In [ ]:
from soyfc.collectors import weather

s.raw["weather"]["points"] = s.raw["weather"]["points"][:2]      # trim for a quick demo
s.raw["weather"]["climatology_start"] = "2023-06-01"             # trim history for a quick demo

wdf = weather.collect(s)
print("weather_daily rows staged:", len(wdf))
wdf.head()

### The other structured collectors (optional)

Same interface. `markets` (yfinance, keyless, a bit slow), `usda_psd` (downloads a ~MB USDA zip,
keyless), `mandi` (needs the `mandi_api`/`datagov` source). Uncomment to run.

In [ ]:
# from soyfc.collectors import markets, usda_psd, mandi
# markets.collect(s)      # CBOT futures, FX, crude, palm/soy oil
# usda_psd.collect(s)     # global soybean balance sheets
# mandi.collect(s)        # target + peer mandi daily prices (the forecast target)
print("skipped (uncomment above to run)")

## 3 - The forecast target (mandi prices)

`build` needs **the target mandi's prices** (whichever market is set in `config/settings.yaml` —
Indore by default) as the target `y`. Collecting them live can be slow or flaky (free APIs), so if
nothing is staged yet we drop in a **synthetic DEMO series** - only so you can run the rest of the
notebook end to end. For real training data, run `mandi.collect(s)` above instead.

In [ ]:
import numpy as np, pandas as pd

if store.read(s, "mandi_daily").empty:
    print("No real mandi data found -> staging SYNTHETIC demo prices (NOT real).")
    days = pd.bdate_range("2014-06-01", s.end_date)
    rng = np.random.default_rng(0)
    price = 4500 + np.cumsum(rng.normal(0, 25, len(days)))     # a random-walk stand-in
    demo = pd.DataFrame({
        "obs_date": days, "state": t["state"], "district": t["district"], "market": t["market"],
        "commodity": t["commodity"], "variety": "NA", "min_price": price - 50, "max_price": price + 50,
        "modal_price": price, "arrivals_t": rng.uniform(100, 500, len(days)),
        "source": "datagov", "available_at": days + pd.Timedelta(days=1)})
    store.upsert(s, "mandi_daily", demo, ["obs_date", "market", "variety", "source"])

print("mandi_daily rows:", len(store.read(s, "mandi_daily")))

## 4 - What's staged so far

Same view as `python -m soyfc status`: one cleaned Parquet table per source, each carrying
`obs_date` (the day the value describes) and `available_at` (the earliest day it could be known).
That second date is the whole point - see Section 7.

In [ ]:
for p in sorted(s.path("staged", "x").parent.glob("*.parquet")):
    df = pd.read_parquet(p)
    rng = (f"{pd.to_datetime(df['obs_date']).min().date()} -> {pd.to_datetime(df['obs_date']).max().date()}"
           if "obs_date" in df and len(df) else "")
    print("%-20s %7d rows  %s" % (p.stem, len(df), rng))

## 5 - Stage 3: BUILD the weekly training table

`build_all(s)` reads every staged table, computes one row per week (each feature = "what was
knowable by that week's Saturday cutoff"), bolts on the forward-looking `target_*` labels, and
writes the output files to `data/features/`.

In [ ]:
from soyfc.features.build import build_all

result = build_all(s)
result

## 6 - Inspect the training data it produced

In [ ]:
X = pd.read_parquet(s.path("features", "training_weekly.parquet"))
print("shape (weeks x columns):", X.shape)
print()
print("some feature columns:", [c for c in X.columns if not c.startswith('target_')][:12])
print("label columns       :", [c for c in X.columns if c.startswith('target_')])

# y = this week's price; y_lag1/y_ma4/ret_1w = history; target_y_fwd1 = next week (a LABEL).
cols = ["y", "y_lag1", "y_ma4", "ret_1w"] + [c for c in X.columns if c.startswith("target_y_fwd")][:1]
X[cols].dropna().tail(8)

In [ ]:
# data_dictionary.csv documents every column: its group, source and availability rule.
dd = pd.read_csv(s.path("features", "data_dictionary.csv"))
print("columns documented:", len(dd))
dd.groupby("group").agg(n_columns=("column", "size"), avg_pct_filled=("non_null_pct", "mean")).round(1)

## 7 - Under the hood: the week grid and point-in-time safety

`week_grid(s)` is the skeleton: one row per week, with `cutoff` = the Saturday the forecast is
issued. Every feature is "the latest value whose `available_at <= cutoff`", computed by the
`asof(...)` helper in `features/build.py`. That is what stops the model seeing the future.

The test `src/soyfc/features/test_features.py::test_no_lookahead` proves it: delete data that was
only knowable after a week's cutoff, rebuild, and that week's features are unchanged.

In [ ]:
from soyfc.features.build import week_grid

grid = week_grid(s)
print("One row per week. cutoff = week_end + forecast_lag_days (Saturday).")
grid.tail(3)

## 8 - Stage 2 (optional): the RESEARCH AGENT

A standard LangChain tool-calling agent (one per research area, per time window) searches Tavily,
reads documents, and saves structured records - guarded in code so every record needs a verbatim
quote from a document published inside the window. The window loop lives in `agent/runner.py`.

**Dry run first** - just the plan, no API calls, no cost:

In [ ]:
from soyfc.agent import runner

plan = runner.run(s, ["news_sentiment", "policy_events"], "2024-09-01", "2024-11-30", dry_run=True)
print("planned (area x month) windows:", len(plan), "| upper-bound searches:", int(plan['max_searches'].sum()))
plan.head()

### Live agent run (costs Gemini + Tavily credits)

Needs `TAVILY_API_KEY` and `GOOGLE_API_KEY` in `.env`. Set the flag to `True` to run one small
window; it stages `agent_news_sentiment`, which then feeds the `sentiment` feature group.

In [ ]:
RUN_LIVE_AGENT = False     # <-- set True to spend a few API calls

if RUN_LIVE_AGENT:
    s.raw["agent"]["max_searches_per_task"] = 2        # keep it cheap
    res = runner.run(s, ["news_sentiment"], "2024-11-01", "2024-11-30")
    display(res)
    display(store.read(s, "agent_news_sentiment").head())
else:
    print("Skipped. Set RUN_LIVE_AGENT = True (and keys in .env) to run the live agent.")

## 9 - Where to go next

- **Outputs** are in `data/features/`: `training_weekly.parquet/.csv` (all features + labels),
  `timellm_<market>_<commodity>.csv` (e.g. `timellm_indore_soyabean.csv`; date + covariates + `y`,
  no labels - for Time-LLM/TimesFM), `data_dictionary.csv`, `coverage_report.md`.
- **For real data**, run `mandi.collect(s)`, `markets.collect(s)`, `usda_psd.collect(s)` (Sections 2-3)
  and remove the weather trims, then re-run `build_all(s)`.
- **From a terminal:** `python -m soyfc collect all`, `python -m soyfc agent`,
  `python -m soyfc build`, or `python -m soyfc run-all` (the weekly job).
- **Read the code** in this order: `config.py` -> `store.py` -> one collector + `http.py` ->
  `features/build.py` (the heart) -> `agent/` (tasks -> schemas -> tools -> agent -> runner).